# 03b Model Explainability: SHAP

This notebook explains a configurable trained model using SHAP values.

Default setting:

```text
PROFILE = transfer
FEATURE_SET = tree
```

## Setup

In [ ]:
import sys
sys.path.append('../')

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import shap
except ImportError as exc:
    raise ImportError(
        "SHAP is not installed. Run `python -m pip install shap` in the active virtual environment, "
        "then restart this notebook kernel."
    ) from exc

RANDOM_STATE = 42
SAMPLE_SIZE = 2000
PROFILE = 'transfer'
FEATURE_SET = 'tree'

RESULT_DIR = f'../outputs/results/{PROFILE}'
FIGURE_DIR = f'../outputs/figures/{PROFILE}'
MODEL_PATH = f'{RESULT_DIR}/best_us_model_{FEATURE_SET}.pkl'
DATA_PATH = f'../data/processed/{PROFILE}/us_modeling_ready_{FEATURE_SET}.csv'

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

## Load Model and Data

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    model_bundle = pickle.load(f)

model = model_bundle['model']
model_name = model_bundle['model_name']
feature_names = model_bundle['feature_names']
scaler = model_bundle.get('scaler')

df = pd.read_csv(DATA_PATH)
X_df = df[feature_names].copy()
y = df['potential_label'].copy()

print(f'Model: {model_name}')
print(f'Profile: {PROFILE}')
print(f'Feature set: {FEATURE_SET}')
print(f'Data shape: {df.shape}')
print(f'Feature count: {len(feature_names)}')
print('Label distribution:')
print(y.value_counts().sort_index())

## Sample Data for SHAP

In [ ]:
sample_n = min(SAMPLE_SIZE, len(X_df))
X_sample = X_df.sample(n=sample_n, random_state=RANDOM_STATE)

if scaler is not None:
    X_model_sample = pd.DataFrame(
        scaler.transform(X_sample),
        columns=feature_names,
        index=X_sample.index,
    )
else:
    X_model_sample = X_sample

print(f'SHAP sample shape: {X_model_sample.shape}')

## Compute SHAP Values

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values_raw = explainer.shap_values(X_model_sample)

if isinstance(shap_values_raw, list):
    shap_values_by_class = shap_values_raw
elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    shap_values_by_class = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]
else:
    shap_values_by_class = [shap_values_raw]

print(f'Number of SHAP output groups/classes: {len(shap_values_by_class)}')
for i, values in enumerate(shap_values_by_class):
    print(f'Class/output {i}: {values.shape}')

## Global SHAP Importance

In [ ]:
mean_abs_by_class = [np.abs(values).mean(axis=0) for values in shap_values_by_class]
overall_importance = np.mean(mean_abs_by_class, axis=0)

shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap_overall': overall_importance,
})

for class_idx, class_importance in enumerate(mean_abs_by_class):
    shap_importance[f'mean_abs_shap_class_{class_idx}'] = class_importance

shap_importance = shap_importance.sort_values('mean_abs_shap_overall', ascending=False)
shap_path = f'{RESULT_DIR}/03b_shap_importance_{FEATURE_SET}.csv'
shap_importance.to_csv(shap_path, index=False)

print(f'Saved to {shap_path}')
print(shap_importance.head(20).to_string(index=False))

## Plot Overall SHAP Importance

In [ ]:
top = shap_importance.head(20).sort_values('mean_abs_shap_overall')

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top['feature'], top['mean_abs_shap_overall'], color='steelblue')
ax.set_xlabel('Mean absolute SHAP value')
ax.set_title(f'Overall SHAP Importance ({model_name}, {FEATURE_SET})')
plt.tight_layout()
fig_path = f'{FIGURE_DIR}/03b_shap_overall_bar_{FEATURE_SET}.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

## SHAP Summary Plots by Class

In [ ]:
class_names = {0: 'Saturated', 1: 'Medium', 2: 'High'}

for class_idx, values in enumerate(shap_values_by_class):
    plt.figure()
    shap.summary_plot(values, X_model_sample, max_display=20, show=False)
    class_label = class_names.get(class_idx, f'class_{class_idx}')
    plt.title(f'SHAP Summary: {class_label} ({class_idx}), {FEATURE_SET}')
    plt.tight_layout()
    fig_path = f'{FIGURE_DIR}/03b_shap_summary_class_{class_idx}_{FEATURE_SET}.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {fig_path}')

## Interpretation Notes

- SHAP explains how the trained model uses features for prediction.
- SHAP values are model explanations, not causal effects.
- For multiclass classification, each class has a separate SHAP explanation.
- Positive SHAP values push a prediction toward the class being explained; negative values push it away from that class.